# Pruebas Estadísticas

### 1. Carga de librerias a utilizar

In [2]:
!pip install pandas numpy scipy scikit-posthocs


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


### 2. Cargamos las librerías a utilizar

In [3]:
import numpy as np
import pandas as pd
from scipy import stats

### 3. Carga de datos

In [4]:
df_biare = pd.read_csv("../Limpieza de datos (Practica 1)/biare_limpio_2021_2024.csv")

df_biare['N_ENT'] = df_biare['N_ENT'].astype(str)
df_biare['N_REN'] = df_biare['N_REN'].astype(str)

### 4. Supuesto de normalidad

Utilizamos el supuesto de normalidad para decidir si usamos ANOVA O Kruskal-wallis, usando la prueba de Shapiro-Wilk, sabemos si los resultados de la pregunta 1 de BIARE tiene una distribución normal.

Si el p-valor de la prueba es mayor a 0.05, entonces tiene una distribución normal, si es menor a 0.05 entonces no tiene una distribución normal.

El resultado estadístico entre más cercano este a 1, más normal es la distribución.

In [5]:
# Shapiro-Wilk sobre una muestra de 500 personas, de la pregunta cb_P1
muestra = df_biare['cb_P1'].dropna().sample(500, random_state=42) #quitamos los nulos si hay y tomamos una muestra de 500 personas permitiendo repeticion con random_state
stat, p_valor = stats.shapiro(muestra)

print(f"Estadístico: {stat:.4f}, p-valor: {p_valor:.10f}")

Estadístico: 0.8088, p-valor: 0.0000000000


Como el p-valor es practicamente 0, eso quiere decir que p-valor < 0.05, por lo que no es una distribución normal, utilizaremos Kruskal-Wallis

### 5. Realizamos la prueba de Kruskal-Wallis por año

Realizamos la prueba de Kruskal-Wallis, para ver si hay alguna diferencía entre las respuestas de la satisfacción con la vida entre los años

Si el p-valor es menor a 0.05 entonces sí hay una diferencia significativa entre al menos un año, si el p-valor es mayor a 0.05, entonces no hay una diferencia significativa entre los años.

In [6]:
#cargamos los anios de cuestionarios que aarecen en la muestra
a2021 = df_biare[df_biare['ANIO'] == 2021]['cb_P1'].dropna()
a2022 = df_biare[df_biare['ANIO'] == 2022]['cb_P1'].dropna()
a2023 = df_biare[df_biare['ANIO'] == 2023]['cb_P1'].dropna()
a2024 = df_biare[df_biare['ANIO'] == 2024]['cb_P1'].dropna()

#calculamos el estadistico y el p-valor de la prueba de Kruskal-Wallis
stat, p_valor = stats.kruskal(a2021, a2022, a2023, a2024)

#mostramos el estadistico y el p-valor de la prueba de Kruskal-Wallis
print(f"Estadístico H: {stat:.4f}")
print(f"p-valor: {p_valor:.10f}")

Estadístico H: 119.5684
p-valor: 0.0000000000


Como el p-valor es 0, entonces el p-valor < 0.05, por lo tanto si hay una diferencia entre al menos un año con respecto a la satisfacción con la vida

##### Prueba de Dunn

Realizamos una prueba de Dunn, para saber entre cuales años hay exactamente una diferencia significante

In [7]:
#importamos la libreria scikit_posthocs para realizar la prueba post-hoc de Dunn
import scikit_posthocs as sp

#realizamos la prueba post-hoc de Dunn con ajuste de Bonferroni
resultado_dunn = sp.posthoc_dunn(df_biare, val_col='cb_P1', group_col='ANIO', p_adjust='bonferroni')
print(resultado_dunn)

              2021          2022          2023          2024
2021  1.000000e+00  1.000000e+00  1.000000e+00  2.703482e-16
2022  1.000000e+00  1.000000e+00  1.000000e+00  2.591205e-20
2023  1.000000e+00  1.000000e+00  1.000000e+00  4.907294e-18
2024  2.703482e-16  2.591205e-20  4.907294e-18  1.000000e+00


Podemos observar que 2024 es el que difiere de los demas años, siendo el único que al compararlo con los demás años, da un resultado cercano a 0 en su p-valor, por lo que nuestro hallazgo es que 2024, difiere de los demás años.

### 5. Realizamos la prueba de Kruskal-Wallis por género

Como lo hicimos anteriormente realizamos la prueba de Kruskal-Wallis, pero en esta ocasión la realizamos para comparar la satisfacción con la vida por género.

Si el p-valor de la prueba de Kruskal-Wallis es mayor a 0.05, entonces no hay diferencia entre hombres y mujeres, por el contrario, sí el p-valor es menor a 0.05, entonces si hay diferencia entre hombres y mujeres.

In [10]:
#dividimos los datos por hombres y mujeres
grupo_hombres = df_biare[df_biare['cs_SEX'] == 'Hombre']['cb_P1'].dropna()
grupo_mujeres = df_biare[df_biare['cs_SEX'] == 'Mujer']['cb_P1'].dropna()

#calculamos el estadistico y el p-valor de la prueba de Kruskal-Wallis para hombres y mujeres
stat, p_valor = stats.kruskal(grupo_hombres, grupo_mujeres)

print(f"Estadístico H: {stat:.4f}")
print(f"p-valor: {p_valor:.10f}")

Estadístico H: 24.5412
p-valor: 0.0000007274


Como el p-valor es menor a 0.05, entonces si hay una diferencia con la satisfacción con la vida entre hombres y mujeres

In [11]:
#mostramos el promedio que cada genero
print("Promedio de satisfacción por género:")
print(df_biare.groupby('cs_SEX')['cb_P1'].mean())

print("\nMediana de satisfacción por género:")
print(df_biare.groupby('cs_SEX')['cb_P1'].median())

Promedio de satisfacción por género:
cs_SEX
Hombre    8.477908
Mujer     8.294621
Name: cb_P1, dtype: float64

Mediana de satisfacción por género:
cs_SEX
Hombre    9.0
Mujer     8.0
Name: cb_P1, dtype: float64


Al ver el promedio y la mediana por género, vemos que aunque en el promedio hay diferencia, no es una diferencia tan significativa, por otro lado al verificar la mediana de cada género, vemos una diferencia de un número entre hombres y mujeres, siendo los hombres los de mayor satisfacción con la vida con una calificación de 9, mientras que las mujeres tienen una califiación de 8, aunque realmente no es una diferenciatan grande.

### 5. Realizamos la prueba de Kruskal-Wallis por estado (seguridad ciudadana)

Como lo hicimos con las pruebas pasadas, realizamos la prueba de Kruskal-Wallis para ver si hay una diferencia entre la percepción de la seguridad ciudadana es diferente entre estados

Si el p-valor de la prueba de Kruskal-Wallis es mayor a 0.05, entonces no hay diferencia entre estados, por el contrario, sí el p-valor es menor a 0.05, entonces si hay diferencia entre estados.

In [13]:
#agrupamos por estado en cuanto a la percepcion de la seguridad ciudadana por estado
grupos_estado = [df_biare[df_biare['ENT'] == estado]['cb_P5_7'].dropna()
                  for estado in df_biare['ENT'].unique()]

#calculamos el estadistico y el p-valor de la prueba de Kruskal-Wallis para los estados
stat, p_valor = stats.kruskal(*grupos_estado)

print(f"Estadístico H: {stat:.4f}")
print(f"p-valor: {p_valor:.10f}")

Estadístico H: 607.9875
p-valor: 0.0000000000


Como el p-valor es menor a 0.05, entonces si hay una diferencia con la percepción de la seguridad ciudadana entre estados

### Prueba de Dunn

Realizamos una prueba de Dunn, para saber entre cuales años hay exactamente una diferencia significante

In [16]:
#realizamos la prueba post-hoc de Dunn con ajuste de Bonferroni
resultado_dunn_estado = sp.posthoc_dunn(df_biare, val_col='cb_P5_7', group_col='ENT', p_adjust='bonferroni')

# Convertimos la tabla a una fila por cada conexion de estados para poder filtrar
dunn_largo = resultado_dunn_estado.stack().reset_index()
dunn_largo.columns = ['Estado_A', 'Estado_B', 'p_valor']

# Quitamos las comparaciones de un estado consigo mismo, y los pares duplicados
dunn_largo = dunn_largo[dunn_largo['Estado_A'] < dunn_largo['Estado_B']]

# Filtramos solo los pares con un p-valor < 0.05
significativos = dunn_largo[dunn_largo['p_valor'] < 0.05].sort_values('p_valor')

print(significativos.head(20))

             Estado_A         Estado_B       p_valor
765   San Luis Potosí         Veracruz  5.045832e-22
381          Guerrero         Veracruz  9.790937e-22
349        Guanajuato         Veracruz  2.261925e-21
959          Veracruz        Zacatecas  3.710580e-21
541            México         Veracruz  3.782144e-21
445           Jalisco         Veracruz  5.810533e-20
247          Coahuila  San Luis Potosí  9.119665e-20
235          Coahuila         Guerrero  2.025946e-19
221  Ciudad de México         Veracruz  2.516070e-19
255          Coahuila        Zacatecas  4.204728e-19
234          Coahuila       Guanajuato  1.106960e-18
766   San Luis Potosí          Yucatán  3.148486e-18
240          Coahuila           México  5.946371e-18
382          Guerrero          Yucatán  7.860779e-18
991           Yucatán        Zacatecas  9.223602e-18
509           Morelos         Veracruz  7.355508e-17
237          Coahuila          Jalisco  8.898494e-17
350        Guanajuato          Yucatán  9.2639

In [15]:
print(df_biare.groupby('ENT')['cb_P5_7'].mean().sort_values())

ENT
Zacatecas              3.897810
Guerrero               4.156425
San Luis Potosí        4.208589
Morelos                4.398810
Guanajuato             4.533784
México                 4.874631
Quintana Roo           4.897436
Michoacán              4.907895
Ciudad de México       5.006127
Puebla                 5.009217
Jalisco                5.028637
Tabasco                5.061644
Aguascalientes         5.237288
Baja California        5.245509
Tlaxcala               5.281690
Chiapas                5.367232
Colima                 5.376404
Hidalgo                5.406250
Durango                5.424419
Campeche               5.710145
Nuevo León             5.856639
Oaxaca                 5.904494
Sinaloa                6.029070
Sonora                 6.285714
Chihuahua              6.288591
Nayarit                6.544944
Querétaro              6.562874
Tamaulipas             6.586826
Baja California Sur    6.654676
Coahuila               6.974490
Veracruz               7.009302
Yuca